In [0]:
# Cria o schema da camada Bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

In [0]:
from pyspark.sql.functions import current_timestamp

caminho = "/Volumes/workspace/cinedata/inputs/"

# Mapeamento arquivo
arquivos_tabelas = {
    "movies_info_TMDB_IMDB.csv": "tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "tb_credits_and_tags",
    "movies_reviews.csv": "tb_movies_reviews",
}

for arquivo, tabela in arquivos_tabelas.items():
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .csv(caminho + arquivo)
        .withColumn("ingestion_datetime", current_timestamp())
    )

    df.write.format("delta").mode("append").saveAsTable(f"workspace.bronze.{tabela}")

    print(f"Tabela {tabela} gravada com {df.count()} linhas.")

In [0]:
import requests
from datetime import datetime, timedelta
from pyspark.sql.functions import current_timestamp

# Widgets para parametrizar as datas de consulta
dbutils.widgets.text("data_inicio", "", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data Fim (MM-DD-AAAA)")

data_inicio_param = dbutils.widgets.get("data_inicio")
data_fim_param = dbutils.widgets.get("data_fim")

# Se os widgets estiverem vazios, usa os últimos 7 dias
if not data_inicio_param or not data_fim_param:
    hoje = datetime.today()
    data_inicio_formatada = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
    data_fim_formatada = hoje.strftime("%m-%d-%Y")
else:
    data_inicio_formatada = data_inicio_param
    data_fim_formatada = data_fim_param

print(f"Consultando cotação de {data_inicio_formatada} até {data_fim_formatada}")

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'"
    "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url)
response.raise_for_status()
dados = response.json()["value"]

print(f"{len(dados)} cotações retornadas pela API")

# Grava na Bronze
df_cotacao = spark.createDataFrame(dados).withColumn("ingestion_datetime", current_timestamp())
df_cotacao.write.format("delta").mode("append").saveAsTable("workspace.bronze.tb_cotacao_dolar")

display(df_cotacao)